In [1]:
## READS MNQ-TICK from HGT-Export SC chartbook

In [1]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [2]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [3]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/mnq-tick/iteration3a'

In [6]:
#symbol = 'mnq'
#SEC = 2

#inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ-TICK-OSCILLATOR-6SEC.csv'
#outFile= f'data/mnq-tick-oscillator-6sec.pqt'
inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ-ALL-3SEC.csv'
outFile= f'data/mnq-tick-all-3sec.pqt'

print(inFile, outFile)

/mnt/d/SierraChart/data/EXPORT/MNQ-ALL-3SEC.csv data/mnq-tick-all-3sec.pqt


In [7]:
df = pd.read_csv(inFile)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11021818 entries, 0 to 11021817
Data columns (total 38 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Date           str    
 1   Time           str    
 2   Open           float64
 3   High           float64
 4   Low            float64
 5   Last           float64
 6   Volume         int64  
 7   #ofTrades      int64  
 8   OHLCAvg        float64
 9   HLCAvg         float64
 10  HLAvg          float64
 11  BidVolume      int64  
 12  AskVolume      int64  
 13  Open.1         float64
 14  High.1         float64
 15  Low.1          float64
 16  Last.1         float64
 17  JMA            float64
 18  VEL            float64
 19  ZeroLevel      float64
 20  VEL.1          float64
 21  ZeroLevel.1    float64
 22  RSX            float64
 23  TopLevel       float64
 24  BottomLevel    float64
 25  ZeroLevel.2    float64
 26  RSX.1          float64
 27  TopLevel.1     float64
 28  BottomLevel.1  float64
 29  ZeroLevel.3    float64


In [8]:
# expand stupid SC date like 2026-7-8 to 2026-07-08
parts = df['Date'].astype(str).str.split('-', expand=True)

year  = parts[0]
month = parts[1].str.zfill(2)
day   = parts[2].str.zfill(2)

df['Date_norm'] = year + '-' + month + '-' + day

# join date and time and convert to wall clock datetime64
df['timestamp'] = pd.to_datetime(
    df['Date_norm'] + ' ' + df['Time'].astype(str),
    utc=False,            # keep as wall clock, no timezone
    errors='raise'        # or 'coerce' if you want bad rows as NaT
)

# remove temp columns
df = df.drop(columns=['Date', 'Date_norm', 'Time'])

# make timestamp first column
col = df.pop('timestamp')
df.insert(0, 'timestamp', col)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11021818 entries, 0 to 11021817
Data columns (total 37 columns):
 #   Column         Dtype         
---  ------         -----         
 0   timestamp      datetime64[us]
 1   Open           float64       
 2   High           float64       
 3   Low            float64       
 4   Last           float64       
 5   Volume         int64         
 6   #ofTrades      int64         
 7   OHLCAvg        float64       
 8   HLCAvg         float64       
 9   HLAvg          float64       
 10  BidVolume      int64         
 11  AskVolume      int64         
 12  Open.1         float64       
 13  High.1         float64       
 14  Low.1          float64       
 15  Last.1         float64       
 16  JMA            float64       
 17  VEL            float64       
 18  ZeroLevel      float64       
 19  VEL.1          float64       
 20  ZeroLevel.1    float64       
 21  RSX            float64       
 22  TopLevel       float64       
 23  BottomLevel    f

In [11]:
'''
RangeIndex: 11021818 entries, 0 to 11021817
Data columns (total 38 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Date           str    
 1   Time           str    
 2   Open           float64
 3   High           float64
 4   Low            float64
 5   Last           float64
 6   Volume         int64  
 7   #ofTrades      int64  
 8   OHLCAvg        float64
 9   HLCAvg         float64
 10  HLAvg          float64
 11  BidVolume      int64  
 12  AskVolume      int64  
 13  Open.1         float64
 14  High.1         float64
 15  Low.1          float64
 16  Last.1         float64
 17  JMA            float64
 18  VEL            float64
 19  ZeroLevel      float64
 20  VEL.1          float64
 21  ZeroLevel.1    float64
 22  RSX            float64
 23  TopLevel       float64
 24  BottomLevel    float64
 25  ZeroLevel.2    float64
 26  RSX.1          float64
 27  TopLevel.1     float64
 28  BottomLevel.1  float64
 29  ZeroLevel.3    float64
 30  Momentum       float64
 31  Line           float64
 32  Momentum.1     float64
 33  Line.1         float64
 34  Momentum.2     float64
 35  Momentum.3     float64
 36  Line.2         float64
 37  JMA.1          float64

RangeIndex: 5512771 entries, 0 to 5512770
Data columns (total 23 columns):
 #   Column        Dtype         
---  ------        -----         
 0   timestamp     datetime64[us]
 1   Open          float64       
 2   High          float64       
 3   Low           float64       
 4   Last          float64       
 5   JMA           float64     on HA Last   
 6   VEL           float64     on JMA  
 7   VEL.1         float64     adptive VEL on JMA
 8   TopBand       float64     bollinger on adaptive VEL  
 9   MiddleBand    float64     +  
 10  BottomBand    float64     +  
 11  RSX           float64     on Open (?)  
 12  RSX.1         float64     TICK RSX on Open (?)  
 13  Momentum      float64     on JMA len=2  
 14  Momentum.1    float64     on Momentum len=3  
 15  TopBand.1     float64     bollinger on Momentum.1  
 16  MiddleBand.1  float64     +  
 17  BottomBand.1  float64     +  
 18  Momentum.2    float64     on TICK JMA len=2
 19  Momentum.3    float64     on Momentum.2 len=3  
 20  TopBand.2     float64     bollinger on Momentum.3  
 21  MiddleBand.2  float64     +  
 22  BottomBand.2  float64     +

 #   Column           Dtype         
---  ------           -----         
 0   timestamp        datetime64[us]
 1   Open             float64       
 2   High             float64       
 3   Low              float64       
 4   Last             float64       
 5   JMA              float64       on HA Last   
 6   VEL              float64       on JMA  
 7   adpVEL           float64       adptive VEL on JMA
 8   bolTopAdpVEL     float64       bollinger on adaptive VEL  
 9   bolMidAdpVEL     float64       +  
 10  bolBotAdpVEL     float64       +  
 11  RSX              float64       on Open (?)  
 12  tickRSX          float64       TICK RSX on Open (?)  
 13  jmaD1            float64       momentum on JMA len=2  
 14  jmaD2            float64       momentum on jmaD1 len=3  
 15  bolTopJmaD2      float64       bollinger on jmaD2  
 16  bolMidJmaD2      float64       +  
 17  bolBotJmaD2      float64       +  
 18  tickJmaD1        float64       momentum on TICK JMA len=2
 19  tickJmaD2        float64       on Momentum on tickJmaD1 len=3  
 20  bolTopTickJmaD2  float64       bollinger on tickJmaD2  
 21  bolMidTickJmaD2  float64       +  
 22  bolBotTickJmaD2  float64       +
 23  tickJMA          float64       TICK JMA on HA Last
 '''

df.drop(columns=['Volume','OHLCAvg','HLCAvg','HLAvg','BidVolume','AskVolume', '#ofTrades',
                 'ZeroLevel','ZeroLevel.1','TopLevel','BottomLevel','ZeroLevel.2','TopLevel.1','BottomLevel.1','ZeroLevel.3','Line','Line.1','Line.2'], inplace=True) 

df.rename(columns={
    'Open': 'rawOpen',
    'High': 'rawHigh',
    'Low' : 'rawLow',
    'Last': 'rawLast',

    'Open.1': 'haOpen',
    'High.1': 'haHigh',
    'Low.1': 'haLow',
    'Last.1': 'haLast',
    
    'VEL.1': 'adpVEL',
    'TopBand': 'bolTopAdpVEL',
    'MiddleBand': 'bolMidAdpVEL',
    'BottomBand': 'bolBotAdpVEL',    
    'RSX.1': 'tickRSX',
    'Momentum': 'jmaD1',
    'Momentum.1': 'jmaD2',
    'TopBand.1': 'bolTopJmaD2',
    'MiddleBand.1': 'bolMidJmaD2',
    'BottomBand.1': 'bolBotJmaD2',
    'Momentum.2': 'tickJmaD1',
    'Momentum.3': 'tickJmaD2',
    'TopBand.2': 'bolTopTickJmaD2',
    'MiddleBand.2': 'bolMidTickJmaD2',
    'BottomBand.2': 'bolBotTickJmaD2',
    'JMA.1': 'tickJMA',
}, inplace=True)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11021818 entries, 0 to 11021817
Data columns (total 19 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   rawOpen    float64       
 2   rawHigh    float64       
 3   rawLow     float64       
 4   rawLast    float64       
 5   haOpen     float64       
 6   haHigh     float64       
 7   haLow      float64       
 8   haLast     float64       
 9   JMA        float64       
 10  VEL        float64       
 11  adpVEL     float64       
 12  RSX        float64       
 13  tickRSX    float64       
 14  jmaD1      float64       
 15  jmaD2      float64       
 16  tickJmaD1  float64       
 17  tickJmaD2  float64       
 18  tickJMA    float64       
dtypes: datetime64[us](1), float64(18)
memory usage: 1.6 GB
None
            timestamp  rawOpen   rawHigh    rawLow   rawLast       haOpen    haHigh     haLow      haLast           JMA          VEL  adpVEL        RSX    tickRSX        jmaD1     

In [12]:
print(f'monotonic: {df["timestamp"].is_monotonic_increasing}')

df['date'] = df['timestamp'].dt.normalize()

filtered_days = [
   '2022-01-17', '2022-07-04', '2022-11-24', '2023-01-16',
   '2023-02-20', '2023-04-07', '2023-05-29', '2023-06-19',
   '2023-07-04', '2023-09-04', '2023-11-23', '2024-02-19',
   '2024-05-27', '2024-06-19', '2024-07-04', '2024-11-28',
   '2025-01-09', '2025-07-04', '2025-09-01', '2025-11-27',
   '2026-04-03', '2026-07-09',               '2025-11-28'
]

# before 1168
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print("Unique days before:", len(all_days))

df = df[
    ~df["timestamp"].dt.normalize().isin(pd.to_datetime(filtered_days))
]

# after -23 = 1145
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print(" Unique days after:", len(all_days))

#

print(df.info())
print(df.head())

monotonic: True
Unique days before: 1169
 Unique days after: 1146
<class 'pandas.DataFrame'>
Index: 10907375 entries, 0 to 11021817
Data columns (total 20 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   rawOpen    float64       
 2   rawHigh    float64       
 3   rawLow     float64       
 4   rawLast    float64       
 5   haOpen     float64       
 6   haHigh     float64       
 7   haLow      float64       
 8   haLast     float64       
 9   JMA        float64       
 10  VEL        float64       
 11  adpVEL     float64       
 12  RSX        float64       
 13  tickRSX    float64       
 14  jmaD1      float64       
 15  jmaD2      float64       
 16  tickJmaD1  float64       
 17  tickJmaD2  float64       
 18  tickJMA    float64       
 19  date       datetime64[us]
dtypes: datetime64[us](2), float64(18)
memory usage: 1.7 GB
None
            timestamp  rawOpen   rawHigh    rawLow   rawLast       haOpen    haHigh     

In [13]:
df.to_parquet(outFile, index=False)
print(f'written to: {outFile}')

In [18]:
print(df["timestamp"].dt.time.min())    
print(f'dates range: {df["date"].min()} .. {df["date"].max()}')    
x = 1/0

08:00:00
dates range: 2022-01-03 00:00:00 .. 2026-07-10 00:00:00


ZeroDivisionError: division by zero

In [14]:
# find out why one is larger than the other by 1 day
outFile1= f'data/mnq-tick-full-3sec.pqt'
outFile2= f'data/mnq-ohlc-raw-3sec.pqt'

df1 = pd.read_parquet(outFile1)
df2 = pd.read_parquet(outFile2)

days1 = df1['date'].unique()
days2 = df2['date'].unique()

print(len(days1), len(days2))

missing = [x for x in days2 if x not in days1]
print(missing)

1145 1145
[]


In [20]:
## WHY 3STREAM DONT MATCH BETWEEN OLD AND NEW
fsrc1 = f'data/mnq-tick-all-3sec.pqt'
fsrc2 = f'data/mnq-tick-full-3sec.pqt'
fraw = f'data/mnq-ohlc-raw-3sec.pqt'

src1 = pd.read_parquet(fsrc1)
src2 = pd.read_parquet(fsrc2)
raw  = pd.read_parquet(fraw)


In [24]:
pd.set_option('display.float_format', lambda x: f'{x:.8f}')

print(src1.head(100))
display(src2.head())
display(raw.head())

             timestamp        rawOpen        rawHigh         rawLow        rawLast         haOpen         haHigh          haLow         haLast            JMA            VEL       adpVEL          RSX       tickRSX          jmaD1          jmaD2    tickJmaD1     tickJmaD2      tickJMA       date
0  2022-01-03 08:00:00 16431.00000000 16431.00000000 16429.75000000 16430.25000000 16431.00000000 16431.00000000 16429.75000000 16430.50000000 24483.94531250  -177.23074341   0.00000000 -99.62631226  -49.54401779     0.00000000     0.00000000   0.00000000    0.00000000 162.84436035 2022-01-03
1  2022-01-03 08:00:03 16430.00000000 16430.75000000 16429.00000000 16429.25000000 16430.75000000 16430.75000000 16429.00000000 16429.75000000 20254.45507813  -459.24676514   0.00000000 -99.86355591  -22.21926498     0.00000000     0.00000000   0.00000000    0.00000000 314.16140747 2022-01-03
2  2022-01-03 08:00:06 16429.50000000 16430.00000000 16429.25000000 16429.25000000 16430.25000000 16430.25000000 16429

,timestamp,Open,High,Low,Last,JMA,VEL,adpVEL,bolTopAdpVEL,bolMidAdpVEL,bolBotAdpVEL,RSX,tickRSX,jmaD1,jmaD2,bolTopJmaD2,bolMidJmaD2,bolBotJmaD2,tickJmaD1,tickJmaD2,bolTopTickJmaD2,bolMidTickJmaD2,bolBotTickJmaD2,tickJMA,date
0,2022-01-03 08:00:00,16431.00000000,16431.00000000,16429.75000000,16430.50000000,24487.08593750,-176.23469543,0.00000000,0.00000000,0.00000000,0.00000000,-99.28514862,-46.13560104,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,188.34887695,2022-01-03
1,2022-01-03 08:00:03,16430.75000000,16430.75000000,16429.00000000,16429.75000000,20255.62695313,-458.66458130,0.00000000,0.00000000,0.00000000,0.00000000,-99.71918488,-27.28374863,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,266.11584473,2022-01-03
2,2022-01-03 08:00:06,16430.25000000,16430.25000000,16429.25000000,16429.50000000,17902.07617188,-739.60644531,0.00000000,0.00000000,0.00000000,0.00000000,-99.83817291,-27.28374863,-6585.00976563,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,266.11584473,2022-01-03
3,2022-01-03 08:00:09,16429.87500000,16430.00000000,16428.75000000,16429.31250000,16847.18359375,-959.01745605,0.00000000,0.00000000,0.00000000,0.00000000,-99.89021301,-13.55491543,-3408.44335938,-3408.44335938,-3074.02075195,-3408.44335938,-3742.86596680,126.99923706,0.00000000,0.00000000,0.00000000,0.00000000,315.34811401,2022-01-03
4,2022-01-03 08:00:12,16429.59375000,16429.75000000,16429.25000000,16429.43750000,16451.35351563,-1119.56250000,0.00000000,0.00000000,0.00000000,0.00000000,-99.91879272,-13.55491543,-1450.72265625,-1450.72265625,-2065.04980469,-2429.58300781,-2794.11621094,126.99923706,0.00000000,0.00000000,0.00000000,0.00000000,315.34811401,2022-01-03


,timestamp,Open,High,Low,Last,OHLCAvg,HLCAvg,HLAvg,date
0,2022-01-03 08:00:00,16431.00000000,16431.00000000,16429.75000000,16430.25000000,16430.50000000,16430.33000000,16430.38000000,2022-01-03
1,2022-01-03 08:00:03,16430.00000000,16430.75000000,16429.00000000,16429.25000000,16429.75000000,16429.67000000,16429.88000000,2022-01-03
2,2022-01-03 08:00:06,16429.50000000,16430.00000000,16429.25000000,16429.25000000,16429.50000000,16429.50000000,16429.63000000,2022-01-03
3,2022-01-03 08:00:09,16429.00000000,16430.00000000,16428.75000000,16429.50000000,16429.31000000,16429.42000000,16429.38000000,2022-01-03
4,2022-01-03 08:00:12,16429.50000000,16429.75000000,16429.25000000,16429.25000000,16429.44000000,16429.42000000,16429.50000000,2022-01-03
